# D03｜金融市场与链上数据导入、清洗、日期匹配与质量检查（v02 正式版）
**负责人**：邓杰良（数据收集、清洗与数据质量负责人）  
**输出文件**：`03_Cleaned_Data/D02_merged_clean_v02.csv`、`cleaning_log_v02.csv`、`tx_extreme_jumps_v02.csv`（以及可选的 2010–2015 历史复现样本 `historical_2010_2015_clean_v02.csv`）

### v02 修正要点
1. **完整日历日期缺口检查**：针对 7×24 连续序列（`BTC_Price_USD`、`BTC_Tx_Count`）检查缺失的公历日期（如 2025-11-13 至 15 日），保留缺失不插值。
2. **区分区间日历天数与实测链上天数**：拆分为 `Calendar_Days_In_Interval`、`Observed_TX_Days` 与 `Interval_Complete`。
3. **如实记录 FTSE 100 数据来源**：标注为 `Yahoo Finance (^FTSE Close Price Index)`。
4. **导出 32 次 >50% 链上交易变动预警明细**：生成 `tx_extreme_jumps_v02.csv` 供人工核查。
5. **支持 2010–2015 原论文样本期**：若 `FTSE100_Raw.csv` 覆盖 2010-07-19 至 2015-05-22，自动同步输出原论文样本期清洗数据集。


In [1]:
# === 1. 环境初始化与路径配置 ===
import os
import json
import pathlib
import numpy as np
import pandas as pd

BASE_DIR = pathlib.Path('.')
RAW_DIR = BASE_DIR / '02_Raw_Data'
CLEAN_DIR = BASE_DIR / '03_Cleaned_Data'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# 主样本区间（2016-01-01 至 2026-10-31）
SAMPLE_START = '2016-01-01'
SAMPLE_END = '2026-10-31'

# 原论文样本区间（Dyhrberg 2016：2010-07-19 至 2015-05-22）
HIST_START = '2010-07-19'
HIST_END = '2015-05-22'

print(f'RAW_DIR:   {RAW_DIR.resolve()}')
print(f'CLEAN_DIR: {CLEAN_DIR.resolve()}')


RAW_DIR:   E:\科大\python\must-btc-hedging-report\02_Raw_Data
CLEAN_DIR: E:\科大\python\must-btc-hedging-report\03_Cleaned_Data


In [2]:
# === 2. 定义各数据源标准化读取函数 ===
def find_raw_file(raw_dir: pathlib.Path, stem: str, exts=('.json', '.csv', '.xlsx', '.xls')) -> pathlib.Path:
    for ext in exts:
        p = raw_dir / f'{stem}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'未找到 {stem} 对应的原始文件（支持扩展名: {exts}），请检查 {raw_dir}')

def load_blockchain_series(file_path: pathlib.Path, value_col_name: str) -> pd.DataFrame:
    suffix = file_path.suffix.lower()
    if suffix == '.json':
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        records = None
        if isinstance(data, dict):
            if 'values' in data and isinstance(data['values'], list):
                records = data['values']
            else:
                for k, v in data.items():
                    if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict) and 'x' in v[0] and 'y' in v[0]:
                        records = v
                        break
        elif isinstance(data, list):
            records = data
        if records is None:
            raise ValueError(f'无法识别的 Blockchain.com JSON 结构: {file_path}')
        df = pd.DataFrame(records)
        ts_sample = pd.to_numeric(df['x'], errors='coerce').dropna().iloc[0]
        unit = 'ms' if ts_sample > 1e12 else 's'
        df['Date'] = pd.to_datetime(df['x'], unit=unit, utc=True).dt.strftime('%Y-%m-%d')
        df[value_col_name] = pd.to_numeric(df['y'], errors='coerce')
        df = df[['Date', value_col_name]]
    else:
        df = pd.read_csv(file_path)
        date_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
        val_col = [c for c in df.columns if c != date_col][-1]
        df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
        df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
        df = df[['Date', value_col_name]]
    return df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

def load_fred_csv(file_path: pathlib.Path, value_col_name: str) -> pd.DataFrame:
    df = pd.read_csv(file_path, na_values=['.', 'NA', 'NaN', ''])
    date_col = [c for c in df.columns if 'date' in c.lower()][0]
    val_col = [c for c in df.columns if c != date_col][0]
    df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
    df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
    return df[['Date', value_col_name]].dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

def load_ftse_file(file_path: pathlib.Path, value_col_name: str = 'FTSE100_Level') -> pd.DataFrame:
    df_raw = pd.read_excel(file_path) if file_path.suffix.lower() in ('.xlsx', '.xls') else pd.read_csv(file_path)
    df = df_raw.copy()
    date_candidates = [c for c in df.columns if any(k in str(c).lower() for k in ['date', 'time', '日期'])]
    date_col = date_candidates[0] if date_candidates else df.columns[0]
    val_candidates = [c for c in df.columns if c != date_col and any(k in str(c).lower() for k in ['close', 'price', 'index', 'ftse', 'pi', 'level'])]
    val_col = val_candidates[0] if val_candidates else df.columns[1]
    df['Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d')
    df[value_col_name] = pd.to_numeric(df[val_col], errors='coerce')
    return df[['Date', value_col_name]].dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)


In [3]:
# === 3. 单序列质量检查与完整日历缺口检查 ===
def run_pipeline_for_period(btc_p_raw, btc_tx_raw, ftse_raw, usdeur_raw, usdgbp_raw, start_d, end_d, is_main=True):
    logs = []
    def inspect_series(df, col_name, source_name, check_full_calendar=False):
        df = df.copy()
        raw_rows = len(df)
        dup_count = int(df['Date'].duplicated().sum())
        if dup_count > 0:
            df = df.drop_duplicates(subset=['Date'], keep='last')
        df_sub = df[(df['Date'] >= start_d) & (df['Date'] <= end_d)].copy()
        na_count = int(df_sub[col_name].isna().sum())
        non_pos_count = int((df_sub[col_name] <= 0).sum())
        df_clean = df_sub.dropna(subset=[col_name])
        df_clean = df_clean[df_clean[col_name] > 0].sort_values('Date').reset_index(drop=True)

        missing_cal_dates = []
        if check_full_calendar and len(df_clean) > 0:
            full_cal = pd.date_range(start_d, end_d, freq='D').strftime('%Y-%m-%d')
            existing = set(df_clean['Date'])
            missing_cal_dates = [d for d in full_cal if d not in existing]

        pct_chg = df_clean[col_name].pct_change()
        extreme_mask = pct_chg.abs() > 0.50
        extreme_count = int(extreme_mask.sum())

        if is_main and col_name == 'BTC_Tx_Count' and extreme_count > 0:
            jumps_df = pd.DataFrame({
                'Prev_Date': df_clean['Date'].shift(1)[extreme_mask],
                'Prev_Value': df_clean[col_name].shift(1)[extreme_mask],
                'Cur_Date': df_clean['Date'][extreme_mask],
                'Cur_Value': df_clean[col_name][extreme_mask],
                'Pct_Change_pct': (pct_chg[extreme_mask] * 100).round(2),
            })
            jumps_df['Calendar_Gap_Days'] = (
                pd.to_datetime(jumps_df['Cur_Date']) - pd.to_datetime(jumps_df['Prev_Date'])
            ).dt.days
            jumps_df.to_csv(CLEAN_DIR / 'tx_extreme_jumps_v02.csv', index=False, encoding='utf-8-sig')

        logs.append({
            'Variable': col_name,
            'Source': source_name,
            'Raw_Rows_All': raw_rows,
            'Actual_Start': df_clean['Date'].min() if len(df_clean) else None,
            'Actual_End': df_clean['Date'].max() if len(df_clean) else None,
            'Sample_Rows_Valid': len(df_clean),
            'Duplicate_Dates_Removed': dup_count,
            'Missing_NA_Dropped': na_count,
            'Non_Positive_Dropped': non_pos_count,
            'Missing_Calendar_Days_Count': len(missing_cal_dates) if check_full_calendar else 'N/A (Trading Days)',
            'Missing_Calendar_Dates': '; '.join(missing_cal_dates[:20]) if missing_cal_dates else '',
            'Extreme_Jump_gt50pct_Flagged': extreme_count,
        })
        return df_clean

    btc_p = inspect_series(btc_p_raw, 'BTC_Price_USD', 'Blockchain.com Market Price (USD)', True)
    btc_t = inspect_series(btc_tx_raw, 'BTC_Tx_Count', 'Blockchain.com Confirmed Transactions Per Day', True)
    ftse  = inspect_series(ftse_raw, 'FTSE100_Level', 'Yahoo Finance (^FTSE Close Price Index)', False)
    eur   = inspect_series(usdeur_raw, 'USD_per_EUR', 'FRED DEXUSEU', False)
    gbp   = inspect_series(usdgbp_raw, 'USD_per_GBP', 'FRED DEXUSUK', False)

    merged = (
        btc_p.merge(ftse, on='Date', how='inner')
        .merge(eur, on='Date', how='inner')
        .merge(gbp, on='Date', how='inner')
        .sort_values('Date').reset_index(drop=True)
    )
    if len(merged) == 0:
        return merged, pd.DataFrame(logs)

    btc_t_sorted = btc_t.sort_values('Date').reset_index(drop=True)
    common_dates = merged['Date'].tolist()
    common_dt = pd.to_datetime(common_dates)
    tx_map = dict(zip(btc_t_sorted['Date'], btc_t_sorted['BTC_Tx_Count']))

    tx_same, tx_mean, tx_sum, cal_days, obs_days = [], [], [], [], []
    for i, cur_d in enumerate(common_dates):
        tx_same.append(tx_map.get(cur_d, np.nan))
        if i == 0:
            cal_days.append(1)
            obs_days.append(1 if cur_d in tx_map else 0)
            tx_mean.append(tx_map.get(cur_d, np.nan))
            tx_sum.append(tx_map.get(cur_d, np.nan))
        else:
            prev_d = common_dates[i - 1]
            cal_gap = int((common_dt[i] - common_dt[i - 1]).days)
            mask = (btc_t_sorted['Date'] > prev_d) & (btc_t_sorted['Date'] <= cur_d)
            sub_vals = btc_t_sorted.loc[mask, 'BTC_Tx_Count']
            cal_days.append(cal_gap)
            obs_days.append(int(len(sub_vals)))
            tx_mean.append(sub_vals.mean() if len(sub_vals) > 0 else np.nan)
            tx_sum.append(sub_vals.sum() if len(sub_vals) > 0 else np.nan)

    merged['BTC_Tx_Count'] = tx_same
    merged['BTC_Tx_Count_Mean'] = tx_mean
    merged['BTC_Tx_Count_Sum'] = tx_sum
    merged['Calendar_Days_In_Interval'] = cal_days
    merged['Observed_TX_Days'] = obs_days
    merged['Interval_Complete'] = (merged['Observed_TX_Days'] == merged['Calendar_Days_In_Interval'])

    for p_col, r_col in [
        ('BTC_Price_USD', 'r_BTC'),
        ('FTSE100_Level', 'r_FTSE'),
        ('USD_per_EUR', 'r_USDEUR'),
        ('USD_per_GBP', 'r_USDGBP'),
    ]:
        merged[r_col] = 100.0 * (np.log(merged[p_col]) - np.log(merged[p_col].shift(1)))

    merged['dln_BTC_Tx_Count'] = 100.0 * (np.log(merged['BTC_Tx_Count']) - np.log(merged['BTC_Tx_Count'].shift(1)))
    merged['dln_BTC_Tx_Count_Mean'] = 100.0 * (np.log(merged['BTC_Tx_Count_Mean']) - np.log(merged['BTC_Tx_Count_Mean'].shift(1)))

    assert merged['Date'].is_unique, "共同日期存在重复值！"
    assert merged['Date'].is_monotonic_increasing, "日期未升序排列！"
    return merged, pd.DataFrame(logs)

btc_price_raw = load_blockchain_series(find_raw_file(RAW_DIR, 'BTC_Market_Price_Raw'), 'BTC_Price_USD')
btc_tx_raw    = load_blockchain_series(find_raw_file(RAW_DIR, 'BTC_Transaction_Count_Raw'), 'BTC_Tx_Count')
ftse_raw      = load_ftse_file(find_raw_file(RAW_DIR, 'FTSE100_Raw', ('.xlsx', '.xls', '.csv')), 'FTSE100_Level')
usdeur_raw    = load_fred_csv(find_raw_file(RAW_DIR, 'USD_EUR_Raw', ('.csv',)), 'USD_per_EUR')
usdgbp_raw    = load_fred_csv(find_raw_file(RAW_DIR, 'USD_GBP_Raw', ('.csv',)), 'USD_per_GBP')

# 运行主样本 (2016-01-01 ~ 2026-10-31)
merged_main, log_main = run_pipeline_for_period(
    btc_price_raw, btc_tx_raw, ftse_raw, usdeur_raw, usdgbp_raw, SAMPLE_START, SAMPLE_END, is_main=True
)
merged_main.to_csv(CLEAN_DIR / 'D02_merged_clean_v02.csv', index=False, encoding='utf-8-sig')
log_main.to_csv(CLEAN_DIR / 'cleaning_log_v02.csv', index=False, encoding='utf-8-sig')
print(f"已导出正式建模数据: 03_Cleaned_Data/D02_merged_clean_v02.csv ({len(merged_main)} 行, {len(merged_main.columns)} 列)")
print(log_main[['Variable', 'Actual_Start', 'Actual_End', 'Sample_Rows_Valid', 'Missing_Calendar_Days_Count', 'Extreme_Jump_gt50pct_Flagged']].to_string(index=False))

# 若 FTSE100_Raw 包含 2010-2015 数据，自动生成原论文样本期 (2010-07-19 ~ 2015-05-22)
if ftse_raw['Date'].min() <= '2010-08-01':
    merged_hist, log_hist = run_pipeline_for_period(
        btc_price_raw, btc_tx_raw, ftse_raw, usdeur_raw, usdgbp_raw, HIST_START, HIST_END, is_main=False
    )
    merged_hist.to_csv(CLEAN_DIR / 'historical_2010_2015_clean_v02.csv', index=False, encoding='utf-8-sig')
    print(f"已导出原论文复现样本: 03_Cleaned_Data/historical_2010_2015_clean_v02.csv ({len(merged_hist)} 行)")
else:
    print("提示：当前 FTSE100_Raw.csv 起始日期为", ftse_raw['Date'].min(), "，若需 2010-2015 原论文样本期，请用 yfinance 从 2010-07-19 起拉取 ^FTSE。")


已导出正式建模数据: 03_Cleaned_Data/D02_merged_clean_v02.csv (2645 行, 17 列)
     Variable Actual_Start Actual_End  Sample_Rows_Valid Missing_Calendar_Days_Count  Extreme_Jump_gt50pct_Flagged
BTC_Price_USD   2016-01-01 2026-10-31               3957                           0                             0
 BTC_Tx_Count   2016-01-01 2026-10-31               3954                           3                            32
FTSE100_Level   2016-01-04 2026-10-30               2736          N/A (Trading Days)                             0
  USD_per_EUR   2016-01-04 2026-10-30               2706          N/A (Trading Days)                             0
  USD_per_GBP   2016-01-04 2026-10-30               2706          N/A (Trading Days)                             0
提示：当前 FTSE100_Raw.csv 起始日期为 2016-01-04 ，若需 2010-2015 原论文样本期，请用 yfinance 从 2010-07-19 起拉取 ^FTSE。
